## Proyecto MathIA
### Clasificación desbalanceada para el diagnóstico de diferencias cognitivas y de aprendizaje en la enseñanza de las matemáticas
#### Transformación de datos

Este programa realiza los siguientes pasos:

- Carga un conjunto de datos en crudo en formato .csv con múltiples muestras por estudiante.
- Describe en pantalla el conjunto de datos en crudo.
- Transforma el conjunto de datos en crudo en un conjunto de datos con una única muestra por estudiante, preparado para clasificación supervisada.
- Muestra en pantalla las 5 primeras filas del conjunto de datos transformado.
- Guarda el conjunto de datos transformado en un fichero .csv.

**Detalles en:**

Ana F. Hernández, Andrej Franulic, Javier A. Salazar, José Palma, Fernando Jiménez (2025), Imbalanced Classification for Diagnosing Learning and Cognitive Differences in Mathematics Education, XI Conference - European Congress of Methodology (EAM2025), La Laguna, Santa Cruz de Tenerife, Spain

##### Autor:
Fernando Jiménez Barrionuevo

In [1]:
import pandas as pd

# Cargar datos (ajusta el nombre del fichero si es necesario)
file_name = 'raw_data.csv'
df = pd.read_csv(file_name)

# Describir
description = df.describe(include='all')
print("DESCRIPCIÓN "+file_name)
print(description)

# Codificar la dificultad como ordinal
dificultad_map = {'D0': 0, 'D1': 1, 'D2': 2, 'D3': 3}
df['DificultadNum'] = df['Dificultad'].map(dificultad_map)

# Diccionario del número de ejercicios propuestos por curso
propuestos_por_curso = {1: 36, 2: 54, 3: 54, 4: 39}

# Añadir columna con el número de ejercicios propuestos según el curso
df["EjerciciosPropuestos"] = df["Curso"].map(propuestos_por_curso)

# Agrupación por estudiante (Identificador)
df_agg = df.groupby("Identificador").agg({
    'Curso': 'first',
    'Sexo(M=1)': 'first',
    'Repetidor': 'first',
    'NotaMates': 'first',
    'MatesPendientes': 'first',
    'TiempoRespuesta': ['mean', 'std', 'min', 'max'],
    'DificultadNum': ['mean', 'std', 'min', 'max'],
    'TipoEjercicio': pd.Series.nunique,
    'CriterioEvaluacion': pd.Series.nunique,
    'EjerciciosPropuestos': 'first',
    'Diagnostico': 'first'
}).reset_index()

# Renombrar columnas para aplanar nombres múltiples niveles
df_agg.columns = ['_'.join(col).strip('_') for col in df_agg.columns.values]

# Calcular suma de ejercicios realizados y correctos por estudiante
realizado_por_estudiante = df.groupby("Identificador")["Realizado"].sum().reset_index(name="TotalRealizados")
correcto_por_estudiante = df.groupby("Identificador")["Correcto"].sum().reset_index(name="TotalCorrectos")

# Unir al DataFrame agregado
df_agg = df_agg.merge(realizado_por_estudiante, on="Identificador")
df_agg = df_agg.merge(correcto_por_estudiante, on="Identificador")

# Calcular proporciones usando el número de ejercicios propuestos
df_agg["ProporcionRealizados"] = df_agg["TotalRealizados"] / df_agg["EjerciciosPropuestos_first"]
df_agg["ProporcionCorrectos"] = df_agg["TotalCorrectos"] / df_agg["EjerciciosPropuestos_first"]

# Eliminar columnas auxiliares si ya no se necesitan
df_agg = df_agg.drop(columns=["EjerciciosPropuestos_first", "TotalRealizados", "TotalCorrectos"])

# Mover la columna 'Diagnostico_first' al final
col_diag = df_agg.pop('Diagnostico_first')
df_agg['Diagnostico'] = col_diag  # Renombrar de paso si quieres

# Mostrar primeras filas del dataframe transformado
print()
print("DATOS TRANSFORMADOS (5 PRIMERAS FILAS) transformed_data.csv")
print(df_agg.head())

# Guardar en CSV
df_agg.to_csv("transformed_data.csv", index=False)

print("\nEjecución completada correctamente.")

DESCRIPCIÓN raw_data.csv
              Curso  Identificador    Sexo(M=1)     TipoEjercicio  \
count   9133.000000   9.133000e+03  9133.000000              9133   
unique          NaN            NaN          NaN                14   
top             NaN            NaN          NaN  Potenciasyraices   
freq            NaN            NaN          NaN              1455   
mean       2.435125   6.577949e+06     0.518888               NaN   
std        0.985262   2.109998e+06     0.499670               NaN   
min        1.000000   4.081484e+06     0.000000               NaN   
25%        2.000000   5.466978e+06     0.000000               NaN   
50%        2.000000   6.292065e+06     1.000000               NaN   
75%        3.000000   6.733097e+06     1.000000               NaN   
max        4.000000   1.383245e+07     1.000000               NaN   

       CriterioEvaluacion NumEjercicio Dificultad  TiempoRespuesta  \
count                9133         9133       9133      9133.000000   
unique